# Day 20 · 构建客服领域评测集

**配套讲义**: [`days/day-20.md`](../days/day-20.md) ｜ **本地可跑，不需要 GPU**

从真实商品与对话里构造 **300+ 条**客服评测样本，分 4 个难度层，并写一份标注手册 —— 这是整个项目**最重要的一份资产**。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w4.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 生成评测集

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.build_domain_eval",
                    "--source", "data/processed/clean.jsonl",
                    "--n", "320", "--out", "data/eval/cx_eval_v1.jsonl", "--card"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2500:] or r.stderr[-2500:])

## 2. 逐层看样本：L1 vs L4 的差距有多大

In [ ]:
import json
from pathlib import Path
from collections import Counter

p = Path("../data/eval/cx_eval_v1.jsonl")
if p.exists():
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print("总数:", len(rows))
    print("难度分布:", Counter(r.get("tier") for r in rows))
    print("意图分布:", Counter(r.get("intent") for r in rows))
    for tier in ("L1", "L2", "L3", "L4"):
        sample = next((r for r in rows if r.get("tier") == tier), None)
        if sample:
            print("=" * 70)
            print(f"[{tier}] {sample.get('question')}")
            print("  必须包含:", sample.get("must_contain"))
            print("  不能出现:", sample.get("must_not_contain"))
else:
    print("先跑上面的生成格子")

## 3. 今天的真正作业：改配额，改模板

自动生成的题目**一定有你看着别扭的**。挑 10 条改成你想要的问法，
把改动写进 `src/eval/build_domain_eval.py` 的模板里，重跑一次。
**这一小时是今天最值钱的一小时。**

In [ ]:
my_edits = """
我改了哪几条：
为什么改：
改完之后哪一层的题目质量明显变好了：
"""
print(my_edits)

## 验收清单

- [ ] `cx_eval_v1.jsonl` 含 300±50 条，**4 个难度层都有量**，8 类意图全部覆盖
- [ ] `python -m src.eval.build_domain_eval --selftest` 全绿
- [ ] `EVAL_CARD.md` 里有泄漏检查结果（必须为 0）和**冻结声明**
- [ ] 标注手册写清了「怎么判对」—— 关键是同义表达算对
- [ ] 隔一周自己重标 30 条，一致性 ≥ 85%（这条需要留时间做）

**卡住了？** 回看 [`days/day-20.md`](../days/day-20.md) 第五节「容易踩的坑」。

> **明天**：`days/day-21.md` —— 自动评测流水线 + LLM-as-judge 校准